# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**

## Why Label Quality Matters
Machine learning models are heavily dependent on the quality and consistency of their training data. If labels are noisy, inconsistent, or invalid, the model's performance metrics will not reflect its true capability.

## Relationship Between EDA and Relabeling
In the previous stage (`02_eda.ipynb`), we identified critical label formatting issues, such as `indotoxic2024` annotations being stored as stringified arrays (e.g., `['0', '0']`). The purpose of this Relabeling notebook is to resolve these specific, documented issues.

## Traceability
To maintain research integrity, original labels MUST be preserved. No label will be permanently overwritten. Every change will be logged in an audit trail.


# 2. RELABELING OBJECTIVES

Our goals for this stage are to:
- Improve label consistency across datasets.
- Align all dataset labels with the official taxonomy (`normal`, `insult`, `harassment`, `threat`, `hate_speech`).
- Safely parse complex label formats (like string arrays).
- Preserve original labels in a separate column.
- Create a transparent audit trail of all changes.

The objective is **NOT** to artificially manipulate the dataset to improve model scores.


In [1]:
# 3. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import pathlib
import ast
import warnings

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
DATA_MERGED_DIR = PROJECT_ROOT / "data" / "merged"
DATA_RELABELED_DIR = PROJECT_ROOT / "data" / "relabeled"
BASE_REPORTS_DIR = PROJECT_ROOT / "reports"\nREPORTS_DIR = BASE_REPORTS_DIR / "relabeling"

MERGED_DATA_PATH = DATA_MERGED_DIR / "merged_cyberbullying_dataset.csv"
RELABELED_DATA_PATH = DATA_RELABELED_DIR / "relabeled_cyberbullying_dataset.csv"
AUDIT_TRAIL_PATH = REPORTS_DIR / "relabeling_audit.csv"
REVIEW_QUEUE_PATH = DATA_RELABELED_DIR / "relabeling_review_queue.csv"

# Columns
TEXT_COL = "text"
TYPE_COL = "cyberbullying_type"
SEVERITY_COL = "severity_level"
ORIGINAL_TYPE_COL = "original_cyberbullying_type"
ORIGINAL_SEVERITY_COL = "original_severity_level"

# Ensure output directories exist
DATA_RELABELED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Merged Data Path: {MERGED_DATA_PATH}")


Merged Data Path: /home/zapp/Kampus/PM-NEW/data/merged/merged_cyberbullying_dataset.csv


In [3]:
# 5. LOAD MERGED DATASET

if not MERGED_DATA_PATH.exists():
    raise FileNotFoundError(f"Merged dataset not found at {MERGED_DATA_PATH}. Please run 01_data_merging.ipynb first.")

df = pd.read_csv(MERGED_DATA_PATH)
print(f"Dataset Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df.head())


Dataset Shape: (44017, 6)
Columns: ['text', 'cyberbullying_type', 'severity_level', 'source_dataset', 'source_file', 'original_row_id']


,text,cyberbullying_type,severity_level,source_dataset,source_file,original_row_id
0,setiap orang adalah seorang gadis yang akan me...,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,0
1,bahwa pos ab kpop stans pergi ke sekolah bersa...,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,1
2,karena beberapa orang tidak ada yang lebih bai...,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,2
3,bro aku pelatih jv tahun lalu di skyline dan a...,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,3
4,wanitawanita ini benarbenar mengingatkan saya ...,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,4


In [4]:
# 6. LOAD EDA FINDINGS

# We rely on insights found during 02_eda.ipynb:
# 1. cyberbullying_cleaned_indo contains labels: 'age', 'ethnicity', 'gender', 'religion', 'not_cyberbullying'
# 2. indotoxic2024 contains stringified arrays: "['0', '0']", "['1', '1']", etc.
# 3. data.csv contains clean labels: 'normal', 'hate_speech', 'insult'

print("EDA Findings successfully integrated into the logic plan.")


EDA Findings successfully integrated into the logic plan.


In [5]:
# 7. INSPECT CURRENT LABEL TAXONOMY

print("Current Unique Labels in Dataset:")
label_dist = df[TYPE_COL].value_counts().reset_index()
label_dist.columns = ['Label', 'Count']
label_dist['Percentage (%)'] = (label_dist['Count'] / len(df)) * 100
display(label_dist.head(15)) # Showing top 15 to avoid wall of text from array labels


Current Unique Labels in Dataset:


,Label,Count,Percentage (%)
0,['0'],15059,34.211782
1,"['0', '0']",5955,13.528864
2,normal,5860,13.313038
3,hate_speech,5561,12.633755
4,insult,1748,3.971193
5,"['0', '0', '0']",1311,2.978395
6,"['1', '0']",913,2.074199
7,"['0', '0', '0', '0']",698,1.585751
8,['1'],689,1.565304
9,"['0', '1']",561,1.274508


# 8. DEFINE OFFICIAL LABEL TAXONOMY

Based on `docs/data-requirements.md` and `docs/problem-definition.md`, the dataset must classify text into:

### Cyberbullying Type

1. **`normal`**: Text that does not contain cyberbullying behavior.
2. **`insult`**: Direct insults, derogatory expressions, or offensive language.
3. **`harassment`**: Repeated, targeted, aggressive, or abusive behavior.
4. **`threat`**: Expression of intention to cause harm, violence, or consequences.
5. **`hate_speech`**: Attacking or expressing hostility based on protected characteristics (age, ethnicity, gender, religion).

### Severity Level
*(Not strictly enforced yet, left as NaN or preserved)*
- `low`, `medium`, `high`


# 9. DEFINE RELABELING RULES

| Rule ID | Description | Condition | Action | Confidence | Review Required |
|---|---|---|---|---|---|
| R1 | Format Standardization | All string labels | Strip whitespace and lowercase | High | No |
| R2 | cyberbullying_cleaned_indo map | Label is 'not_cyberbullying' | Map to `normal` | High | No |
| R3 | Identity attack map | Label in ['age', 'ethnicity', 'gender', 'religion'] | Map to `hate_speech` | High | No |
| R4 | indotoxic2024 array '0' | Label is stringified array containing only '0's | Map to `normal` | High | No |
| R5 | indotoxic2024 array '1' | Label is stringified array containing '1' | Map to `harassment` | Medium | No |
| R7 | cyberbullying_cleaned_indo map | Label is 'other_cyberbullying' | Map to `harassment` | High | No |
| R6 | Ambiguous / Unknown | Label does not match taxonomy or rules | Preserve label, mark `manual_review_required` | Low | Yes |


In [6]:
# 10. PRESERVE ORIGINAL LABELS

df[ORIGINAL_TYPE_COL] = df[TYPE_COL].copy()

if SEVERITY_COL in df.columns:
    df[ORIGINAL_SEVERITY_COL] = df[SEVERITY_COL].copy()
    
print("Original labels successfully backed up.")


Original labels successfully backed up.


# 10.5 DROP NULL TEXTS

Based on validation findings, there are rows with missing text (null/NaN).
Since text is the most critical feature for NLP, rows without text are useless and must be removed.

In [7]:
initial_len = len(df)
null_text_mask = df[TEXT_COL].isnull()
null_text_count = null_text_mask.sum()

if null_text_count > 0:
    print(f"Dropping {null_text_count} rows with missing text...")
    df = df[~null_text_mask].copy()
    print(f"Dataset size reduced from {initial_len} to {len(df)}.")
else:
    print("No missing texts found.")


Dropping 10 rows with missing text...
Dataset size reduced from 44017 to 44007.


In [8]:
# 11. DETECT CANDIDATE RECORDS

official_labels = ['normal', 'insult', 'harassment', 'threat', 'hate_speech']

candidates_mask = ~df[TYPE_COL].isin(official_labels)
print(f"Found {candidates_mask.sum()} candidate records requiring relabeling.")


Found 30838 candidate records requiring relabeling.


In [9]:
# 12. APPLY SAFE STANDARDIZATION RULES & 13. HANDLE INVALID LABELS

def apply_relabeling(row):
    orig = str(row[ORIGINAL_TYPE_COL])
    new_label = orig
    
    # R1: Formatting
    new_label = new_label.strip().lower()
    
    # R2 & R3: cyberbullying_cleaned_indo
    if new_label == 'not_cyberbullying':
        return pd.Series(['normal', 'R2', 'Mapped not_cyberbullying to normal', 'High', 'Rule-based correction'])
    if new_label == 'other_cyberbullying':
        return pd.Series(['harassment', 'R7', 'Mapped other_cyberbullying to harassment', 'High', 'Rule-based correction'])
        
    if new_label in ['age', 'ethnicity', 'gender', 'religion']:
        return pd.Series(['hate_speech', 'R3', 'Mapped protected characteristic attack to hate_speech', 'High', 'Rule-based correction'])
        
    # R4 & R5: indotoxic array parsing
    if new_label.startswith('[') and new_label.endswith(']'):
        try:
            arr = ast.literal_eval(new_label)
            if all(v == '0' for v in arr):
                return pd.Series(['normal', 'R4', 'Indotoxic array contains only 0s', 'High', 'Rule-based correction'])
            elif any(v == '1' for v in arr):
                return pd.Series(['harassment', 'R5', 'Indotoxic array contains 1', 'Medium', 'Rule-based correction'])
        except (ValueError, SyntaxError):
            pass 
            
    # Existing valid label check
    if new_label in official_labels:
        # If the original had whitespace but now it matches, it's R1
        if orig != new_label:
            return pd.Series([new_label, 'R1', 'Whitespace/lowercase standardization', 'High', 'Automatically standardized'])
        return pd.Series([new_label, 'None', 'Already valid', 'High', 'No change'])

    # R6: Ambiguous
    return pd.Series([orig, 'R6', 'Unknown or ambiguous label', 'Low', 'Manual review required'])

df[['cyberbullying_type', 'rule_id', 'relabel_reason', 'confidence', 'review_status']] = df.apply(apply_relabeling, axis=1)

print("Standardization rules applied.")


Standardization rules applied.


# 13.5 DROP CONFLICTING DUPLICATES

Based on validation findings, some exact texts have conflicting labels (e.g., one row says `normal`, another says `insult` for the exact same text).
Since we cannot safely assume which one is correct without manual review, and they will confuse the model, we will drop all conflicting duplicates to preserve data quality.

In [10]:
initial_len = len(df)
# Find texts that have more than 1 unique label
grouped = df.groupby(TEXT_COL)[TYPE_COL].nunique()
conflicting_texts = grouped[grouped > 1].index

conflicting_mask = df[TEXT_COL].isin(conflicting_texts)
conflicting_count = conflicting_mask.sum()

if conflicting_count > 0:
    print(f"Dropping {conflicting_count} rows with conflicting labels...")
    df = df[~conflicting_mask].copy()
    print(f"Dataset size reduced from {initial_len} to {len(df)}.")
else:
    print("No conflicting duplicates found.")


Dropping 1053 rows with conflicting labels...
Dataset size reduced from 44007 to 42954.


In [11]:
# 14. CREATE RELABELING AUDIT TRAIL

audit_df = df[df['rule_id'] != 'None'][['original_row_id', 'source_dataset', 'source_file', TEXT_COL, ORIGINAL_TYPE_COL, TYPE_COL, 'relabel_reason', 'rule_id', 'confidence', 'review_status']]
print(f"Total records audited (changed or flagged): {len(audit_df)}")
display(audit_df.head())


Total records audited (changed or flagged): 29813


,original_row_id,source_dataset,source_file,text,original_cyberbullying_type,cyberbullying_type,relabel_reason,rule_id,confidence,review_status
0,0,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,setiap orang adalah seorang gadis yang akan me...,age,hate_speech,Mapped protected characteristic attack to hate...,R3,High,Rule-based correction
1,1,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,bahwa pos ab kpop stans pergi ke sekolah bersa...,age,hate_speech,Mapped protected characteristic attack to hate...,R3,High,Rule-based correction
2,2,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,karena beberapa orang tidak ada yang lebih bai...,age,hate_speech,Mapped protected characteristic attack to hate...,R3,High,Rule-based correction
3,3,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,bro aku pelatih jv tahun lalu di skyline dan a...,age,hate_speech,Mapped protected characteristic attack to hate...,R3,High,Rule-based correction
4,4,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,wanitawanita ini benarbenar mengingatkan saya ...,age,hate_speech,Mapped protected characteristic attack to hate...,R3,High,Rule-based correction


In [12]:
# 15. CREATE MANUAL REVIEW QUEUE

review_queue = df[df['review_status'] == 'Manual review required'][['original_row_id', 'source_dataset', 'source_file', TEXT_COL, ORIGINAL_TYPE_COL, TYPE_COL, 'relabel_reason', 'confidence', 'review_status']]

if len(review_queue) > 0:
    review_queue.to_csv(REVIEW_QUEUE_PATH, index=False)
    print(f"Saved {len(review_queue)} records to manual review queue: {REVIEW_QUEUE_PATH}")
else:
    print("No records require manual review. Review queue is empty.")


No records require manual review. Review queue is empty.


In [13]:
# 16. GENERATE RELABELING SUMMARY

print("### Relabeling Summary ###\n")
print(f"Original unique labels: {df[ORIGINAL_TYPE_COL].nunique()}")
print(f"New unique labels: {df[TYPE_COL].nunique()}")

print("\nRule Execution Counts:")
print(df['rule_id'].value_counts())

print("\nFinal Label Distribution:")
print(df[TYPE_COL].value_counts())


### Relabeling Summary ###

Original unique labels: 289
New unique labels: 4

Rule Execution Counts:
rule_id
R4      22676
None    13141
R5       4756
R3       1599
R2        391
R7        391
Name: count, dtype: int64

Final Label Distribution:
cyberbullying_type
normal         28914
hate_speech     7147
harassment      5147
insult          1746
Name: count, dtype: int64


In [14]:
# 17. SAVE RELABELED DATASET

final_cols = ['text', 'cyberbullying_type', 'severity_level', 'original_cyberbullying_type', 'original_severity_level', 'source_dataset', 'source_file', 'original_row_id']
final_cols = [c for c in final_cols if c in df.columns]

relabeled_df = df[final_cols]
relabeled_df.to_csv(RELABELED_DATA_PATH, index=False)

print(f"SUCCESS: Relabeled dataset saved to {RELABELED_DATA_PATH}")
print(f"Final output shape: {relabeled_df.shape}")


SUCCESS: Relabeled dataset saved to /home/zapp/Kampus/PM-NEW/data/relabeled/relabeled_cyberbullying_dataset.csv
Final output shape: (42954, 8)


In [15]:
# 18. SAVE AUDIT ARTIFACTS

if len(audit_df) > 0:
    audit_df.to_csv(AUDIT_TRAIL_PATH, index=False)
    print(f"SUCCESS: Relabeling audit trail saved to {AUDIT_TRAIL_PATH}")


SUCCESS: Relabeling audit trail saved to /home/zapp/Kampus/PM-NEW/reports/relabeling_audit.csv


# 19. HOW TO RUN THIS NOTEBOOK

1. Activate the project's virtual environment.
2. Ensure the merged dataset exists in: `data/merged/merged_cyberbullying_dataset.csv`.
3. Open `notebooks/03_relabeling.ipynb`.
4. Review the official label taxonomy in **Step 8**.
5. Review the relabeling rules in **Step 9**.
6. Run the notebook from the first cell to the last cell.
7. Review the relabeling audit trail in `reports/relabeling_audit.csv`.
8. Review the manual review queue in `data/relabeled/relabeling_review_queue.csv` (if any).
9. Confirm the output exists in `data/relabeled/relabeled_cyberbullying_dataset.csv`.
10. Proceed to `notebooks/04_validation.ipynb`.
